# Generalizing Optimal Arbitrage to N-Hop Paths via Möbius Transformations

This notebook generalizes the closed-form arbitrage formula from the [original single-hop analysis](https://medium.com/@jeffreyhartigan/deriving-a-closed-form-solution-for-optimal-dex-arbitrage-b7c917bbe3e7) to arbitrary multi-hop paths across constant product AMMs. The key insight: each AMM swap is a **Möbius transformation**, and Möbius transformations compose via matrix multiplication. This means the entire n-hop path collapses to a single fractional-linear function, preserving the closed-form structure regardless of path length.

The result is a general formula that computes the profit-maximizing trade size for any n-hop circular arbitrage in $O(n)$ time with **zero iteration** — extending the original $O(1)$ two-pool formula to paths of arbitrary length.

In [1]:
import numpy as np
from numpy import sqrt
import time

## Recap: Single-Hop Closed Form

In the [original analysis](optimal_arbitrage.ipynb), we derived the profit-maximizing input for a two-pool arbitrage. Starting from the constant product equations with fee factor $k$:

$$y = b - \frac{ab}{a + kx}, \quad z = c - \frac{cd}{d + ky}$$

We found that setting $\frac{dz}{dx} = 1$ and solving for $x$ yields:

$$x_{\text{ideal}} = \frac{k\sqrt{abcd} - ad}{k(bk + d)}$$

where $(a, b)$ and $(c, d)$ are pool reserves and $k = 0.997$ is the fee factor. This is an exact $O(1)$ result — no iteration, no approximation.

**The question: can this be generalized to n hops?**

The original writeup noted that multi-hop paths produce "higher-degree polynomials under the square root" and "may not admit clean closed-form solutions." It turns out this concern was unfounded. The algebra has hidden structure that keeps the solution clean for any number of hops.

## The Key Insight: AMM Swaps as Möbius Transformations

A single constant product swap with input reserve $r$, output reserve $s$, and fee factor $f$ produces:

$$\text{output} = \frac{s \cdot f \cdot \text{input}}{r + f \cdot \text{input}}$$

This is algebraically equivalent to the formula $y = b - \frac{ab}{a + kx}$ from the original derivation:

$$y = b - \frac{ab}{a + kx} = \frac{b(a + kx) - ab}{a + kx} = \frac{bkx}{a + kx}$$

This function has the form $\frac{\alpha x}{\beta + \gamma x}$, which is a **Möbius transformation** (also called a linear fractional transformation), specifically one that fixes the origin and maps $0 \to 0$.

Möbius transformations have a remarkable algebraic property: **the composition of any two Möbius transformations is again a Möbius transformation.** Furthermore, because every AMM swap sends $0 \to 0$ (zero input always gives zero output), the composed function after any number of hops retains this special form:

$$l(x) = \frac{Kx}{M + Nx}$$

where $K$, $M$, $N$ are constants determined solely by the pool reserves and fees along the path. No matter how many hops — 2, 5, 50 — the output function always has this same simple structure.

## Matrix Representation

Each hop can be encoded as a $2 \times 2$ matrix. The Möbius transformation $f(x) = \frac{sf \cdot x}{r + f \cdot x}$ corresponds to:

$$\mathbf{M}_i = \begin{pmatrix} s_i f_i & 0 \\ f_i & r_i \end{pmatrix}$$

The matrix acts on $x$ via the rule: if $\mathbf{M} = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$, then $f(x) = \frac{ax + b}{cx + d}$.

Composing $n$ hops is simply **matrix multiplication** (applied right-to-left, matching function composition order):

$$\mathbf{M}_{\text{total}} = \mathbf{M}_n \cdot \mathbf{M}_{n-1} \cdots \mathbf{M}_2 \cdot \mathbf{M}_1 = \begin{pmatrix} K & 0 \\ N & M \end{pmatrix}$$

The top-right entry remains exactly zero throughout the multiplication — a consequence of every individual map fixing the origin ($0 \to 0$).

The composed output is then:

$$l(x) = \frac{Kx}{M + Nx}$$

This is the entire multi-hop transfer function encoded in three numbers.

## The 3-Number Recurrence

The matrix product can be computed without storing full matrices. We maintain just three scalars $K$, $M$, $N$ and update them one hop at a time.

**Initialize** with the first hop (input reserve $r_1$, output reserve $s_1$, fee $f_1$):

$$K = s_1 f_1, \quad M = r_1, \quad N = f_1$$

**Update** for each subsequent hop $i$ with reserves $(r_i, s_i)$ and fee $f_i$:

$$K_{\text{new}} = s_i \cdot f_i \cdot K$$

$$M_{\text{new}} = r_i \cdot M$$

$$N_{\text{new}} = r_i \cdot N + f_i \cdot K$$

**Important:** the update for $N$ uses the value of $K$ **before** it is updated in this step. In code, all three values are computed from the old state before any are overwritten.

After processing all $n$ hops, we have the final $K$, $M$, $N$ and the composed transfer function $l(x) = \frac{Kx}{M + Nx}$.

## General Derivative and Optimal Arbitrage

With $l(x) = \frac{Kx}{M + Nx}$, the derivative is straightforward:

$$\frac{dl}{dx} = \frac{K(M + Nx) - Kx \cdot N}{(M + Nx)^2} = \frac{KM}{(M + Nx)^2}$$

The profit function is $\text{profit}(x) = l(x) - x$, so the optimality condition is:

$$\frac{d}{dx}\text{profit}(x) = \frac{dl}{dx} - 1 = 0$$

$$\frac{KM}{(M + Nx)^2} = 1$$

$$(M + Nx)^2 = KM$$

$$M + Nx = \sqrt{KM} \quad \text{(positive root)}$$

$$\boxed{x_{\text{opt}} = \frac{\sqrt{KM} - M}{N}}$$

This formula works for **any number of hops**. Profitable arbitrage exists when $\frac{K}{M} > 1$ (the initial marginal exchange rate exceeds unity).

**Complexity:** $O(n)$ to compute $K, M, N$ via the recurrence (one pass through the pools), then $O(1)$ for the square root formula. The total is $O(n)$ — linear in the number of hops, with no iteration or numerical solving.

In [2]:
def compose_amm_chain(pools):
    """
    Compute K, M, N for a multi-hop AMM path using the 3-number recurrence.

    Parameters:
        pools: list of tuples (r_in, s_out, fee_factor)
               Each tuple represents one hop in the path.

    Returns:
        K, M, N such that the composed output is l(x) = Kx / (M + Nx)
    """
    r, s, f = pools[0]
    K = s * f
    M = r
    N = f

    for r, s, f in pools[1:]:
        K_new = s * f * K
        M_new = r * M
        N_new = r * N + f * K  # uses K before update
        K, M, N = K_new, M_new, N_new

    return K, M, N


def optimal_arbitrage_multihop(pools):
    """
    Compute the profit-maximizing input for an n-hop arbitrage path.

    Parameters:
        pools: list of tuples (r_in, s_out, fee_factor)

    Returns:
        x_opt: optimal input amount (0 if no profitable arbitrage)
        profit: expected profit at x_opt
    """
    K, M, N = compose_amm_chain(pools)

    # No arbitrage if initial marginal rate <= 1
    if K <= M:
        return 0.0, 0.0

    x_opt = (sqrt(K * M) - M) / N

    # Compute actual profit: l(x_opt) - x_opt
    l_x = K * x_opt / (M + N * x_opt)
    profit = l_x - x_opt

    return x_opt, profit


def simulate_multihop(x, pools):
    """Simulate the actual output of sending x through the multi-hop path."""
    amount = x
    for r, s, f in pools:
        amount = (s * f * amount) / (r + f * amount)
    return amount

## Verification: 2-Hop Case Matches the Original Formula

Using the same SHIB/ETH pool data from the original analysis, the multi-hop recurrence should produce identical results to the single-hop formula $x_{\text{ideal}} = \frac{k\sqrt{abcd} - ad}{k(bk + d)}$.

To map the 2-pool setup into the multi-hop framework:
- **Hop 1:** Input token₀ into Pool A. Input reserve = $a$ (token₀), output reserve = $b$ (token₁), fee = $k$.
- **Hop 2:** Input token₁ into Pool B. Input reserve = $d$ (token₁), output reserve = $c$ (token₀), fee = $k$.

The recurrence gives:
- After hop 1: $K = bk$, $M = a$, $N = k$
- After hop 2: $K = bck^2$, $M = ad$, $N = k(bk + d)$

Plugging into $x_{\text{opt}} = \frac{\sqrt{KM} - M}{N}$:

$$x_{\text{opt}} = \frac{\sqrt{bck^2 \cdot ad} - ad}{k(bk + d)} = \frac{k\sqrt{abcd} - ad}{k(bk + d)}$$

This is **exactly** the original formula.

In [3]:
# Original pool data from the single-hop analysis (SHIB/ETH on Uniswap V2)
a = 15800.025178893529930149   # Token0 in Pool A
b = 30348149.556699            # Token1 in Pool A
c = 5251.705779226172996106    # Token0 in Pool B
d = 9986593.845926             # Token1 in Pool B
k = 0.997                     # Fee factor (0.3% fee)

# Original closed-form formula (single-hop specific)
x_original = (k * sqrt(a * b * c * d) - a * d) / (k * (b * k + d))

# Multi-hop recurrence with 2 pools
# Hop 1: token0 -> token1 via Pool A  (r=a, s=b, f=k)
# Hop 2: token1 -> token0 via Pool B  (r=d, s=c, f=k)
pools_2hop = [(a, b, k), (d, c, k)]
x_multihop, profit_multihop = optimal_arbitrage_multihop(pools_2hop)

# Compare
print("2-Hop Verification Against Original Formula")
print("=" * 50)
print(f"Original formula:     x = {x_original:.10f}")
print(f"Multi-hop recurrence: x = {x_multihop:.10f}")
print(f"Difference:               {abs(x_original - x_multihop):.2e}")
print(f"\nProfit (analytic):        {profit_multihop:.10f}")

# Cross-check via simulation
actual_output = simulate_multihop(x_multihop, pools_2hop)
print(f"Simulated output:         {actual_output:.10f}")
print(f"Simulated profit:         {actual_output - x_multihop:.10f}")
print(f"Profit match:             {abs(profit_multihop - (actual_output - x_multihop)) < 1e-10}")

2-Hop Verification Against Original Formula
Original formula:     x = 7.9210884160
Multi-hop recurrence: x = 7.9210884160
Difference:               8.88e-16

Profit (analytic):        0.0159546619
Simulated output:         7.9370430779
Simulated profit:         0.0159546619
Profit match:             True


## Multi-Hop Arbitrage: 3, 4, and 5 Hop Paths

To validate the general formula beyond the 2-hop case, we construct synthetic multi-hop circular arbitrage paths. Each path represents a round-trip trade (e.g., Token A $\to$ B $\to$ C $\to$ A) where price discrepancies across pools create a profitable cycle.

For each path, we verify that:
1. The closed-form $x_{\text{opt}} = \frac{\sqrt{KM} - M}{N}$ produces a valid result
2. The analytic profit matches the simulated profit (feeding $x_{\text{opt}}$ through the actual swap chain)
3. The initial marginal rate $K/M > 1$ (confirming arbitrage exists)

In [4]:
# 3-hop path: Token A -> Token B -> Token C -> Token A
# Reserves chosen to create a small round-trip price edge
pools_3hop = [
    (1000.0, 2050000.0, 0.997),    # Pool 1: A/B  (r_in=A, s_out=B)
    (2000000.0, 1020.0, 0.997),    # Pool 2: B/C  (r_in=B, s_out=C)
    (980.0, 1060.0, 0.997),        # Pool 3: C/A  (r_in=C, s_out=A)
]

# 4-hop path: Token A -> B -> C -> D -> A
pools_4hop = [
    (5000.0, 10200000.0, 0.997),   # Pool 1: A/B
    (10000000.0, 5100.0, 0.997),   # Pool 2: B/C
    (4900.0, 9900000.0, 0.997),    # Pool 3: C/D
    (9500000.0, 5250.0, 0.997),    # Pool 4: D/A
]

# 5-hop path: Token A -> B -> C -> D -> E -> A
pools_5hop = [
    (10000.0, 20500000.0, 0.997),  # Pool 1: A/B
    (20000000.0, 10200.0, 0.997),  # Pool 2: B/C
    (9800.0, 20000000.0, 0.997),   # Pool 3: C/D
    (19500000.0, 10100.0, 0.997),  # Pool 4: D/E
    (9700.0, 10500.0, 0.997),      # Pool 5: E/A
]

# Compute and verify for each path length
for name, pools in [("3-hop", pools_3hop), ("4-hop", pools_4hop), ("5-hop", pools_5hop)]:
    K, M, N = compose_amm_chain(pools)
    marginal_rate = K / M
    x_opt, profit = optimal_arbitrage_multihop(pools)

    # Verify via simulation
    actual_output = simulate_multihop(x_opt, pools)
    actual_profit = actual_output - x_opt

    print(f"\n{name} path:")
    print(f"  K/M (initial marginal rate): {marginal_rate:.6f}")
    print(f"  x_opt:                       {x_opt:.6f}")
    print(f"  Analytic profit:             {profit:.6f}")
    print(f"  Simulated profit:            {actual_profit:.6f}")
    print(f"  Match:                       {abs(profit - actual_profit) < 1e-10}")


3-hop path:
  K/M (initial marginal rate): 1.120700
  x_opt:                       19.078671
  Analytic profit:             1.118604
  Simulated profit:            1.118604
  Match:                       True

4-hop path:
  K/M (initial marginal rate): 1.147772
  x_opt:                       85.827103
  Analytic profit:             6.123027
  Simulated profit:            6.123027
  Match:                       True

5-hop path:
  K/M (initial marginal rate): 1.178442
  x_opt:                       162.151803
  Analytic profit:             13.873792
  Simulated profit:            13.873792
  Match:                       True


## The Reciprocal Coordinate: a Two-Scalar Recurrence

Taking the reciprocal of a swap makes it affine. With $u = 1/x$:

$$\frac{1}{\text{output}} = \frac{r + f x}{s f x} = \alpha u + \beta, \qquad \alpha = \frac{r}{f s}, \quad \beta = \frac{1}{s}$$

Composing affine maps needs only two numbers. Start from $(A, B) = (\alpha_1, \beta_1)$ and for each later hop apply

$$A \leftarrow \alpha_i A, \qquad B \leftarrow \alpha_i B + \beta_i$$

The composed path is $l(x) = \dfrac{x}{A + Bx}$, with $A = M/K$ and $B = N/K$. The optimum and its profit are

$$x_{\text{opt}} = \frac{\sqrt{A} - A}{B}, \qquad \text{profit}_{\max} = \frac{(1 - \sqrt{A})^2}{B}$$

and arbitrage exists when $A < 1$.

**Why this matters numerically.** $K$ and $M$ are products of raw reserves, so they grow geometrically with path length and $K \cdot M$ overflows: in float32 after about five hops with reserves near $10^7$, in float64 after a few dozen. $A$ is a product of exchange-rate ratios, which around a cycle multiply to a number near 1, so the $(A, B)$ form does not grow with path length. The $(K, M, N)$ form is still the right one for exact integer arithmetic because it has no division.

In [5]:
def compose_affine(pools):
    """
    Two-scalar recurrence in the reciprocal coordinate.

    Returns A, B such that the composed output is l(x) = x / (A + B x).
    """
    A, B = 1.0, 0.0
    for r, s, f in pools:
        alpha = r / (f * s)
        A, B = alpha * A, alpha * B + 1.0 / s
    return A, B


def optimal_arbitrage_affine(pools):
    """Optimal input and profit from the (A, B) recurrence. Returns (0, 0) if no arbitrage."""
    A, B = compose_affine(pools)
    if A >= 1.0:
        return 0.0, 0.0
    root = sqrt(A)
    return (root - A) / B, (1.0 - root) ** 2 / B


print(f"{'Path':<8} {'x_opt (K,M,N)':>18} {'x_opt (A,B)':>18} {'rel. diff':>12}")
print("-" * 60)
for name, pools in [("2-hop", pools_2hop), ("3-hop", pools_3hop), ("4-hop", pools_4hop), ("5-hop", pools_5hop)]:
    x_kmn, _ = optimal_arbitrage_multihop(pools)
    x_ab, _ = optimal_arbitrage_affine(pools)
    print(f"{name:<8} {x_kmn:>18.10f} {x_ab:>18.10f} {abs(x_kmn - x_ab) / x_kmn:>12.1e}")

Path          x_opt (K,M,N)        x_opt (A,B)    rel. diff
------------------------------------------------------------
2-hop          7.9210884160       7.9210884160      4.9e-14
3-hop         19.0786708259      19.0786708259      5.6e-16
4-hop         85.8271032750      85.8271032750      1.7e-16
5-hop        162.1518032568     162.1518032569      2.5e-15


## Comparison Against Iterative Solvers

A fair comparison needs a fair baseline, so the closed form is measured against two iterative methods that do **not** use the composed form:

1. **Bisection with a finite-difference derivative.** The simple approach: bisect on the sign of a numerically estimated profit derivative.
2. **Newton's method with exact derivatives.** The first and second derivatives of the path output are propagated hop by hop with the chain rule, and Newton's method is applied to $l'(x) - 1 = 0$ starting from $x = 0$. Since $l'$ is convex and decreasing, the iteration converges monotonically from that start.

Accuracy is measured against a 50-digit reference computed with Python's `decimal` module.

In [6]:
from decimal import Decimal, getcontext
import timeit

getcontext().prec = 50


def reference_optimum(pools):
    """50-digit reference for x_opt, computed from the exact K, M, N."""
    r, s, f = (Decimal(float(v)) for v in pools[0])
    K, M, N = s * f, r, f
    for hop in pools[1:]:
        r, s, f = (Decimal(float(v)) for v in hop)
        K, M, N = s * f * K, r * M, r * N + f * K
    return ((K * M).sqrt() - M) / N


def multihop_profit(x, pools):
    """Profit for input x through a multi-hop path."""
    return simulate_multihop(x, pools) - x


def bisection_fd(pools, precision=1e-12, max_iter=1000):
    """Bisection on a central finite-difference estimate of the profit derivative."""
    left, right = 1e-12, min(r for r, s, f in pools)
    iterations = 0
    for _ in range(max_iter):
        mid = (left + right) / 2
        eps = max(abs(mid) * 1e-8, 1e-12)
        dp = (multihop_profit(mid + eps, pools) - multihop_profit(mid - eps, pools)) / (2 * eps)
        if (right - left) < precision:
            break
        if dp > 0:
            left = mid
        else:
            right = mid
        iterations += 1
    return mid, iterations


def path_derivatives(x, pools):
    """Output, first and second derivative of the path at x, by the chain rule."""
    y, d1, d2 = x, 1.0, 0.0
    for r, s, f in pools:
        den = r + f * y
        g1 = s * f * r / den ** 2
        g2 = -2.0 * s * f * f * r / den ** 3
        y, d1, d2 = s * f * y / den, g1 * d1, g2 * d1 * d1 + g1 * d2
    return y, d1, d2


def newton_exact(pools, rtol=1e-12, max_iter=100):
    """Newton's method on l'(x) - 1 = 0 from x = 0, using exact derivatives."""
    x = 0.0
    for it in range(1, max_iter + 1):
        _, d1, d2 = path_derivatives(x, pools)
        step = (d1 - 1.0) / d2
        x -= step
        if abs(step) <= rtol * abs(x):
            break
    return x, it


def time_us(fn, repeat=5):
    """Best-of-`repeat` wall time per call, in microseconds."""
    timer = timeit.Timer(fn)
    number, _ = timer.autorange()
    return min(timer.repeat(repeat=repeat, number=number)) / number * 1e6

In [7]:
all_pools = [("2-hop", pools_2hop), ("3-hop", pools_3hop), ("4-hop", pools_4hop), ("5-hop", pools_5hop)]

print(f"{'Path':<7} {'Method':<24} {'x_opt':>16} {'rel. error':>12} {'Time (us)':>11} {'Iterations':>11}")
print("-" * 86)
for name, pools in all_pools:
    ref = reference_optimum(pools)
    rows = [
        ("Bisection (finite diff.)", lambda: bisection_fd(pools)),
        ("Newton (exact deriv.)", lambda: newton_exact(pools)),
        ("Closed form (K, M, N)", lambda: (optimal_arbitrage_multihop(pools)[0], 0)),
        ("Closed form (A, B)", lambda: (optimal_arbitrage_affine(pools)[0], 0)),
    ]
    for label, fn in rows:
        x, iters = fn()
        err = abs((Decimal(float(x)) - ref) / ref)
        print(f"{name:<7} {label:<24} {x:>16.10f} {float(err):>12.1e} {time_us(fn):>11.2f} {iters:>11}")
    print()

Path    Method                              x_opt   rel. error   Time (us)  Iterations
--------------------------------------------------------------------------------------


2-hop   Bisection (finite diff.)     7.9211005984      1.5e-06       26.08          54


2-hop   Newton (exact deriv.)        7.9210884160      6.8e-14        2.36           4


2-hop   Closed form (K, M, N)        7.9210884160      4.0e-14        0.51           0


2-hop   Closed form (A, B)           7.9210884160      8.3e-15        0.51           0



3-hop   Bisection (finite diff.)    19.0786698162      5.3e-08       30.47          50


... (7 outputs left out)

5-hop   Bisection (finite diff.)   162.1518022409      6.3e-09       41.71          54


5-hop   Newton (exact deriv.)      162.1518032569      6.6e-16        5.38           5


5-hop   Closed form (K, M, N)      162.1518032568      1.0e-15        0.79           0


5-hop   Closed form (A, B)         162.1518032569      1.4e-15        0.58           0



What the table shows:

- **Accuracy.** Both closed forms and Newton agree with the reference to near machine precision. Bisection with a finite-difference derivative is the least accurate: the profit curve is flat at its maximum, so the estimated derivative is dominated by rounding noise before the bracket has closed. Its *profit* is still essentially optimal for the same reason.
- **Speed.** Against bisection the closed form is faster by roughly the iteration count. Against Newton, which needs only a handful of iterations, the gap is a single-digit multiple.
- **The durable advantage** is not the constant factor. The closed form has no iteration count, no tolerance, no bracket and no starting point, and its cost is one pass over the pools.

Timings are pure-Python wall times on whatever machine ran this notebook, so only the ratios are meaningful.

## Scaling With Path Length, on Economically Consistent Cycles

Random reserves do not make a realistic cycle: if every pool has a small input reserve and a large output reserve, each hop multiplies the amount by a large factor and every path looks absurdly profitable. Instead, cycles are generated from a consistent set of token prices:

- each token $i$ in the cycle gets a reference price $P_i$ (log-uniform over six orders of magnitude);
- pool $i$ trades token $i$ for token $i+1$ and holds value $V_i$ on each side (log-uniform between $10^5$ and $10^7$), so $r_i = V_i / P_i$ and $s_i = V_i / P_{i+1}$;
- each pool's output reserve is then perturbed by a small mispricing $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$.

With no mispricing the marginal rate around the cycle is exactly $\prod_i f_i < 1$, so there is no arbitrage. With mispricing, only some cycles clear the fee hurdle, as in a real market.

In [8]:
def generate_cycles(batch, n_hops, sigma=0.01, fee=0.997, seed=0):
    """
    Generate `batch` price-consistent n-hop cycles.

    Returns an array of shape (batch, n_hops, 3) holding [r_in, s_out, fee_factor].
    """
    rng = np.random.default_rng(seed)
    log_p = rng.uniform(-3.0, 3.0, size=(batch, n_hops))            # log10 token prices
    price_in = 10.0 ** log_p
    price_out = np.roll(price_in, -1, axis=1)                        # token i+1, wrapping to token 0
    value = 10.0 ** rng.uniform(5.0, 7.0, size=(batch, n_hops))      # value per side of each pool
    mispricing = np.exp(rng.normal(0.0, sigma, size=(batch, n_hops)))
    r_in = value / price_in
    s_out = value / price_out * mispricing
    fees = np.full((batch, n_hops), fee)
    return np.stack([r_in, s_out, fees], axis=2)


def profitable_cycle(n_hops, seed=0):
    """Return one profitable cycle of the given length as a list of (r, s, f) tuples."""
    batch = generate_cycles(2000, n_hops, sigma=0.02, seed=seed + n_hops)
    for path in batch:
        pools = [tuple(float(v) for v in hop) for hop in path]
        if compose_affine(pools)[0] < 0.995:
            return pools
    raise RuntimeError("no profitable cycle found")


print(f"{'Hops':>5} {'Closed (A,B) us':>16} {'Newton us':>11} {'iters':>6} {'Bisection us':>13} {'iters':>6} "
      f"{'vs Newton':>10} {'vs Bisect.':>11}")
print("-" * 86)
for n in [2, 3, 4, 5, 6, 8, 10, 15, 20]:
    pools = profitable_cycle(n)
    t_cf = time_us(lambda: optimal_arbitrage_affine(pools))
    t_nw = time_us(lambda: newton_exact(pools))
    t_bs = time_us(lambda: bisection_fd(pools))
    _, it_nw = newton_exact(pools)
    _, it_bs = bisection_fd(pools)
    print(f"{n:>5} {t_cf:>16.2f} {t_nw:>11.2f} {it_nw:>6} {t_bs:>13.2f} {it_bs:>6} "
          f"{t_nw / t_cf:>9.1f}x {t_bs / t_cf:>10.1f}x")

 Hops  Closed (A,B) us   Newton us  iters  Bisection us  iters  vs Newton  vs Bisect.
--------------------------------------------------------------------------------------


    2             0.50        2.35      4         26.55     55       4.7x       52.8x


    3             0.44        2.76      4         26.43     51       6.3x       60.1x


    4             0.56        3.28      4         30.04     51       5.9x       53.5x


    5             0.71        6.05      5         35.51     51       8.6x       50.2x


    6             0.69        5.95      5         44.00     50       8.7x       64.2x


    8             0.71        7.41      4         47.67     51      10.5x       67.3x


   10             0.99       10.42      5         64.41     50      10.6x       65.4x


   15             1.18       15.56      5         78.92     48      13.2x       66.8x


   20             1.54       16.89      5         82.93     49      11.0x       53.8x


## Batch Evaluation: Scanning Many Candidate Cycles at Once

In a production search the bottleneck is not one path but the number of candidate paths that can be evaluated per block. The recurrence is plain arithmetic with no data-dependent control flow, so it vectorizes across a batch: $A$ and $B$ become tensors of shape `(batch,)`, and each hop is one element-wise update over every candidate at once.

Three things are kept honest here:

- **Matched precision.** GPU backends such as MPS only support float32, so the CPU is timed in float32 as well as float64. Comparing float64 on CPU with float32 on GPU would credit the GPU with the precision change.
- **Usable answers.** The batch uses the $(A, B)$ recurrence. The $(K, M, N)$ recurrence overflows float32 on realistic reserves, which the next cell demonstrates.
- **A batched iterative baseline.** Newton's method with exact derivatives is run on the same tensors for a fixed number of iterations, so the speedup from removing iteration is measured on the same device.

In [9]:
try:
    import torch
    TORCH_AVAILABLE = True
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        GPU_DEVICE, GPU_NAME = torch.device("mps"), "MPS (Apple GPU)"
    elif torch.cuda.is_available():
        GPU_DEVICE, GPU_NAME = torch.device("cuda"), f"CUDA ({torch.cuda.get_device_name(0)})"
    else:
        GPU_DEVICE, GPU_NAME = None, "none (CPU-only results below)"
    print(f"PyTorch {torch.__version__}")
    print(f"GPU: {GPU_NAME}")
except ImportError:
    TORCH_AVAILABLE = False
    GPU_DEVICE = None
    print("PyTorch not installed. Install with: pip install torch")
    print("The batch cells below will be skipped.")


def gpu_sync():
    """Block until queued GPU work has finished, so timings are real."""
    if GPU_DEVICE is None:
        return
    if GPU_DEVICE.type == "mps":
        torch.mps.synchronize()
    else:
        torch.cuda.synchronize()

PyTorch 2.15.0+cpu
GPU: none (CPU-only results below)


In [10]:
if TORCH_AVAILABLE:
    def optimal_arbitrage_batched(pools_tensor):
        """
        Closed-form optimum for a batch of paths, via the (A, B) recurrence.

        pools_tensor: shape (batch, n_hops, 3) holding [r_in, s_out, fee_factor].
        Returns x_opt, profit, each of shape (batch,), zero where no arbitrage exists.
        """
        r, s, f = pools_tensor.unbind(dim=2)
        alpha = r / (f * s)
        beta = 1.0 / s
        A = alpha[:, 0].clone()
        B = beta[:, 0].clone()
        for i in range(1, pools_tensor.shape[1]):
            A, B = alpha[:, i] * A, alpha[:, i] * B + beta[:, i]
        root = torch.sqrt(A)
        arb = A < 1.0
        zero = torch.zeros_like(A)
        return torch.where(arb, (root - A) / B, zero), torch.where(arb, (1.0 - root) ** 2 / B, zero)


    def optimal_arbitrage_batched_kmn(pools_tensor):
        """The original (K, M, N) batch recurrence, kept only to show where it overflows."""
        r, s, f = pools_tensor.unbind(dim=2)
        K, M, N = s[:, 0] * f[:, 0], r[:, 0].clone(), f[:, 0].clone()
        for i in range(1, pools_tensor.shape[1]):
            K, M, N = s[:, i] * f[:, i] * K, r[:, i] * M, r[:, i] * N + f[:, i] * K
        return (torch.sqrt(K * M) - M) / N


    def newton_batched(pools_tensor, n_iter=6):
        """Batched Newton on l'(x) - 1 = 0 from x = 0, exact derivatives, fixed iteration count."""
        r, s, f = pools_tensor.unbind(dim=2)
        x = torch.zeros_like(r[:, 0])
        for _ in range(n_iter):
            y, d1, d2 = x, torch.ones_like(x), torch.zeros_like(x)
            for i in range(pools_tensor.shape[1]):
                den = r[:, i] + f[:, i] * y
                g1 = s[:, i] * f[:, i] * r[:, i] / den ** 2
                g2 = -2.0 * s[:, i] * f[:, i] ** 2 * r[:, i] / den ** 3
                y, d1, d2 = s[:, i] * f[:, i] * y / den, g1 * d1, g2 * d1 * d1 + g1 * d2
            x = x - (d1 - 1.0) / d2
        return torch.clamp(x, min=0.0)


    # Accuracy of the batch methods against a float64 reference, on 5-hop cycles
    cycles = generate_cycles(100_000, 5, sigma=0.01, seed=1)
    t64 = torch.tensor(cycles, dtype=torch.float64)
    t32 = t64.to(torch.float32)

    x_ref, profit_ref = optimal_arbitrage_batched(t64)
    arb = x_ref > 0
    clear = profit_ref > 1e-3 * x_ref          # cycles with an edge well clear of the fee hurdle
    print(f"5-hop cycles generated: {len(cycles):,}   profitable: {int(arb.sum()):,} "
          f"({100 * arb.double().mean():.1f}%)")

    def rel_err(x, mask):
        return ((x.double() - x_ref).abs() / x_ref)[mask]

    x32, _ = optimal_arbitrage_batched(t32)
    agree = (x32 > 0) == arb
    print(f"\nfloat32 (A, B) closed form")
    print(f"  same profitable/unprofitable verdict as float64: {100 * agree.double().mean():.3f}% of cycles")
    both = arb & (x32 > 0)
    print(f"  median relative error in x_opt, all profitable:  {rel_err(x32, both).median():.1e}")
    print(f"  max relative error in x_opt, clear-edge cycles:  {rel_err(x32, both & clear).max():.1e}")

    x_kmn32 = optimal_arbitrage_batched_kmn(t32)
    print(f"\nfloat32 (K, M, N) closed form")
    print(f"  non-finite results: {100 * (~torch.isfinite(x_kmn32)).double().mean():.1f}% of cycles")

    x_nw = newton_batched(t64)
    print(f"\nfloat64 Newton, 6 iterations")
    print(f"  max relative error in x_opt, all profitable:     {rel_err(x_nw, arb).max():.1e}")

5-hop cycles generated: 100,000   profitable: 25,072 (25.1%)

float32 (A, B) closed form
  same profitable/unprofitable verdict as float64: 100.000% of cycles
  median relative error in x_opt, all profitable:  1.1e-05
  max relative error in x_opt, clear-edge cycles:  2.3e-04

float32 (K, M, N) closed form
  non-finite results: 99.7% of cycles

float64 Newton, 6 iterations
  max relative error in x_opt, all profitable:     1.5e-09


In [11]:
if TORCH_AVAILABLE:
    def time_batch(fn, tensor, min_time=0.3):
        """Mean wall time per call in microseconds, after warm-up, with GPU sync."""
        for _ in range(3):
            fn(tensor)
        gpu_sync()
        runs, start = 0, time.perf_counter()
        while True:
            fn(tensor)
            gpu_sync()
            runs += 1
            elapsed = time.perf_counter() - start
            if elapsed >= min_time and runs >= 3:
                return elapsed / runs * 1e6

    n_hops = 5
    batch_sizes = [1_000, 10_000, 100_000, 1_000_000]

    header = (f"{'Batch':>10} {'CPU f64 (us)':>14} {'CPU f32 (us)':>14} {'CPU f32 Newton':>16} {'Newton/closed':>14}")
    if GPU_DEVICE is not None:
        header += f" {'GPU f32 (us)':>14} {'GPU vs CPU f32':>15}"
    print(f"{n_hops}-hop cycles, closed form via (A, B) unless marked Newton (6 iterations)\n")
    print(header)
    print("-" * len(header))

    for batch in batch_sizes:
        data = generate_cycles(batch, n_hops, sigma=0.01, seed=2)
        cpu64 = torch.tensor(data, dtype=torch.float64)
        cpu32 = cpu64.to(torch.float32)
        t_cpu64 = time_batch(optimal_arbitrage_batched, cpu64)
        t_cpu32 = time_batch(optimal_arbitrage_batched, cpu32)
        t_newton = time_batch(newton_batched, cpu32)
        row = f"{batch:>10,} {t_cpu64:>14.1f} {t_cpu32:>14.1f} {t_newton:>16.1f} {t_newton / t_cpu32:>13.1f}x"
        if GPU_DEVICE is not None:
            gpu32 = cpu32.to(GPU_DEVICE)
            t_gpu = time_batch(optimal_arbitrage_batched, gpu32)
            row += f" {t_gpu:>14.1f} {t_cpu32 / t_gpu:>14.1f}x"
        print(row)

5-hop cycles, closed form via (A, B) unless marked Newton (6 iterations)

     Batch   CPU f64 (us)   CPU f32 (us)   CPU f32 Newton  Newton/closed
------------------------------------------------------------------------


     1,000           93.2           87.0           1468.3          16.9x


    10,000          524.0          352.9           4666.2          13.2x


   100,000         4170.4         2466.5          45809.3          18.6x


 1,000,000        65735.4        36400.8         602141.6          16.5x


Reading the batch results:

- **Newton / closed** is the cost of iterating, measured on the same device and precision. It is close to the iteration count times the extra work of carrying derivatives, and it is the number that supports the claim that removing iteration lets a searcher scan more candidates per block.
- **GPU vs CPU f32** is the device speedup at matched precision. Small batches are dominated by dispatch overhead, so the GPU only pays off at large batch sizes.
- **float32 caveat.** Near the break-even point $A \approx 1$, the formula subtracts nearly equal numbers, so float32 loses relative accuracy in $x_{\text{opt}}$ for marginal cycles and can flip the profitable/unprofitable verdict for cycles sitting within rounding error of the fee hurdle. Those cycles have negligible profit, and the profit curve is flat at its maximum, so this matters for ranking only at the margin. A production scanner would use float32 on GPU to shortlist and re-evaluate the shortlist in float64 or exact integer arithmetic.

## Conclusion

1. **Each hop** is a linear fractional map fixing the origin, representable as $\mathbf{M}_i = \begin{pmatrix} s_i f_i & 0 \\ f_i & r_i \end{pmatrix}$. In the reciprocal coordinate $u = 1/x$ it is the affine map $u \mapsto \alpha_i u + \beta_i$.

2. **A full path** composes by matrix multiplication, or equivalently by the three-scalar recurrence on $(K, M, N)$ or the two-scalar recurrence on $(A, B)$.

3. **The output** always has the form $l(x) = \dfrac{Kx}{M + Nx} = \dfrac{x}{A + Bx}$, regardless of hop count.

4. **The optimal input** is $x_{\text{opt}} = \dfrac{\sqrt{KM} - M}{N} = \dfrac{\sqrt{A} - A}{B}$, with maximum profit $\dfrac{(1 - \sqrt{A})^2}{B}$.

5. **Numerically**, the $(A, B)$ form is stable in floating point at any path length, while $(K, M, N)$ is the division-free form suited to exact integer arithmetic.

6. **Against iterative solvers**, the closed form matches a 50-digit reference to machine precision, needs no iteration count, tolerance or bracket, and costs one pass over the pools. The measured speedup depends on the baseline: large against bisection, a small multiple against Newton with exact derivatives.

The equivalent-reserves reduction for cyclic arbitrage on constant-product pools appears in Wang et al., "Cyclic Arbitrage in Decentralized Exchanges" (arXiv:2105.02784). The contribution here is the structural view (swaps as affine maps in the reciprocal coordinate) and the recurrences that follow from it.